In [1]:
!pip install langchain-ollama langchain-chroma langchain-huggingface langchain-community chromadb tqdm python-telegram-bot nest-asyncio sentence-transformers pandas -q

import nest_asyncio
nest_asyncio.apply()

# Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

print("⏳ Запуск Ollama...")
# Убиваем старые процессы
!pkill ollama 2>/dev/null || true

# Запускаем сервер
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

print("⏳ Скачивание модели Llama 3.1...")
!ollama pull llama3.1
print("✅ Готово!")

from google.colab import drive
drive.mount('/content/drive')

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
⏳ Запуск Ollama...
⏳ Скачивание модели Llama 3.1...

✅ Готово!
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%writefile /content/config.py
import logging
import os

logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO)
logger = logging.getLogger("HR_Researcher_Bot")

# === ВАШ ТОКЕН ===
TELEGRAM_TOKEN = "8379355384:AAHQVQs1KyQvYtjlzQ0mZvluPzE9yU636eM"
# =================

OLLAMA_MODEL = "llama3.1"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ДАННЫЕ С ДИСКА
DRIVE_BASE = "/content/drive/MyDrive/researcher"
RESUME_FILE = os.path.join(DRIVE_BASE, "researcher_1000samples.csv")

# БАЗА ДАННЫХ
DB_PATH = "/content/chroma_resume_db"
DB_BACKUP_PATH = os.path.join(DRIVE_BASE, "chroma_resume_db_backup.zip")

HR_COLLECTION_NAME = "hr_resumes_v1"
RESUME_LIMIT = 1000

# === СПИСКИ КНОПОК ===
SAMPLE_HR_QUERIES = [
    "Backend разработчик",
    "Frontend разработчик",
    "DevOps разработчик",
    "IT project manager",
    "Data Scientist",
    "Аналитик данных",
    "Тестировщик"
]

Writing /content/config.py


In [3]:
%%writefile /content/keyboards.py
from telegram import ReplyKeyboardMarkup

def get_main_keyboard():
    """Главное меню"""
    keyboard = [
        ["👨‍💻 Найти кандидата"],
        ["ℹ️ Помощь", "🔄 Обновить базу"],
        ["🎲 Случайный запрос"]
    ]
    return ReplyKeyboardMarkup(keyboard, resize_keyboard=True)

def get_cancel_keyboard():
    """Клавиатура для отмены"""
    keyboard = [
        ["❌ Отменить поиск"]
    ]
    return ReplyKeyboardMarkup(keyboard, resize_keyboard=True)

def get_position_keyboard():
    """Выбор должности - ТОЛЬКО указанные вами варианты"""
    keyboard = [
        ["Backend разработчик"],
        ["Frontend разработчик"],
        ["DevOps разработчик"],
        ["IT project manager"],
        ["Другое"]
    ]
    return ReplyKeyboardMarkup(keyboard, resize_keyboard=True)

def get_salary_keyboard():
    """Выбор зарплатного диапазона"""
    keyboard = [
        ["до 50000 ₽", "50000-100000 ₽"],
        ["100000-150000 ₽", "150000-200000 ₽"],
        ["200000-300000 ₽", "выше 300000 ₽"],
        ["Не важно"]
    ]
    return ReplyKeyboardMarkup(keyboard, resize_keyboard=True)

def get_experience_keyboard():
    """Выбор опыта работы"""
    keyboard = [
        ["Нет опыта", "До 1 года"],
        ["1-3 года", "3-5 лет"],
        ["5-10 лет", "Более 10 лет"],
        ["Не важно"]
    ]
    return ReplyKeyboardMarkup(keyboard, resize_keyboard=True)

Writing /content/keyboards.py


In [4]:
%%writefile /content/normalizer.py
import re
import pandas as pd

def clean_text(text):
    """Очищает текст от лишних пробелов и символов"""
    if not text or pd.isna(text):
        return ""
    text = str(text).replace("\xa0", " ").replace("\r", "").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()

def normalize_resume(row):
    """
    Собирает структурированное текстовое представление резюме из CSV строки.
    Использует поля из вашего датасета.
    """
    text_parts = []

    # Заголовок с ID
    if row.get('resume_id'):
        text_parts.append(f"🆔 ID: {row['resume_id']}")

    # Ссылка
    if row.get('url'):
        text_parts.append(f"🔗 Ссылка: {row['url']}")

    # Основная информация
    profession = clean_text(row.get('profession', ''))
    desired_position = clean_text(row.get('desired_position', ''))

    if profession:
        text_parts.append(f"📌 ПРОФЕССИЯ: {profession}")
    if desired_position:
        text_parts.append(f"🎯 ЖЕЛАЕМАЯ ДОЛЖНОСТЬ: {desired_position}")

    # Опыт работы
    work_experience = clean_text(row.get('work_experience_years', ''))
    if work_experience:
        text_parts.append(f"📅 ОПЫТ РАБОТЫ: {work_experience} лет")

    experience_bucket = clean_text(row.get('experience_bucket', ''))
    if experience_bucket:
        text_parts.append(f"📊 УРОВЕНЬ ОПЫТА: {experience_bucket}")

    # Зарплата
    salary = clean_text(row.get('salary_value', ''))
    if salary and row.get('has_salary_value', 'True') == 'True':
        text_parts.append(f"💰 ЗАРПЛАТА: {salary}")

    # Демографические данные
    if row.get('age'):
        text_parts.append(f"🎂 ВОЗРАСТ: {row['age']}")

    if row.get('is_male'):
        gender = "Мужской" if str(row['is_male']).lower() in ['true', '1', 'male'] else "Женский"
        text_parts.append(f"👫 ПОЛ: {gender}")

    # Локация
    location = clean_text(row.get('location', ''))
    if location:
        text_parts.append(f"📍 ЛОКАЦИЯ: {location}")

    # Образование
    education_level = clean_text(row.get('education_level', ''))
    if education_level:
        text_parts.append(f"🎓 ОБРАЗОВАНИЕ: {education_level}")

    education_facility = clean_text(row.get('education_facility', ''))
    if education_facility:
        text_parts.append(f"🏫 УЧЕБНОЕ ЗАВЕДЕНИЕ: {education_facility}")

    # Языки
    languages = clean_text(row.get('languages', ''))
    if languages:
        text_parts.append(f"🗣 ЯЗЫКИ: {languages}")

    english_level = clean_text(row.get('english_level', ''))
    if english_level:
        text_parts.append(f"🌐 АНГЛИЙСКИЙ: {english_level}")

    # Навыки
    skills = clean_text(row.get('skills', ''))
    if skills:
        # Обрезаем слишком длинные навыки
        if len(skills) > 300:
            skills = skills[:300] + "..."
        text_parts.append(f"🛠 НАВЫКИ: {skills}")

    # О себе
    self_description = clean_text(row.get('self_description', ''))
    if self_description:
        if len(self_description) > 400:
            self_description = self_description[:400] + "..."
        text_parts.append(f"📝 О СЕБЕ: {self_description}")

    # Флаг emoji
    if row.get('has_emoji') == 'True':
        text_parts.append("🎭 Есть emoji в резюме")

    return "\n".join(text_parts)

Writing /content/normalizer.py


In [5]:
%%writefile /content/data_loader.py
import json
import csv
import os
import logging
import pandas as pd
from langchain_core.documents import Document
from config import logger, RESUME_FILE, RESUME_LIMIT
from normalizer import normalize_resume

def load_resumes_csv():
    """
    Загружает резюме из CSV файла.
    Возвращает список документов для векторной базы.
    """
    docs = []

    if not os.path.exists(RESUME_FILE):
        logger.warning(f"Файл {RESUME_FILE} не найден!")
        return docs

    try:
        # Читаем CSV с обработкой различных кодировок
        try:
            df = pd.read_csv(RESUME_FILE, encoding='utf-8')
        except UnicodeDecodeError:
            logger.info("Пробую кодировку cp1251...")
            df = pd.read_csv(RESUME_FILE, encoding='cp1251')

        # Ограничиваем количество записей
        df = df.head(RESUME_LIMIT)

        # Заменяем NaN на пустые строки
        df = df.fillna('')

        logger.info(f"✅ CSV загружен: {len(df)} записей")
        logger.info(f"📊 Колонки: {list(df.columns)}")

        # Создаем документы для векторной базы
        for i, row in df.iterrows():
            # Преобразуем ряд в словарь
            row_dict = row.to_dict()

            # Создаем нормализованный текст резюме
            text = normalize_resume(row_dict)

            # Создаем метаданные
            metadata = {
                "source": "csv_resume",
                "id": row_dict.get('resume_id', f"resume_{i}"),
                "profession": str(row_dict.get('profession', '')),
                "desired_position": str(row_dict.get('desired_position', '')),
                "work_experience_years": str(row_dict.get('work_experience_years', '')),
                "salary_value": str(row_dict.get('salary_value', '')),
                "has_salary_value": str(row_dict.get('has_salary_value', 'False')),
                "experience_bucket": str(row_dict.get('experience_bucket', '')),
                "age": str(row_dict.get('age', '')),
                "location": str(row_dict.get('location', '')),
                "education_level": str(row_dict.get('education_level', '')),
                "english_level": str(row_dict.get('english_level', '')),
                "skills": str(row_dict.get('skills', ''))[:200]  # Обрезаем для метаданных
            }

            # Создаем документ
            doc = Document(
                page_content=text,
                metadata=metadata
            )

            docs.append(doc)

        logger.info(f"✅ Документы созданы: {len(docs)}")

        # Показываем пример первого документа для отладки
        if docs:
            logger.info("📝 Пример первого документа:")
            logger.info(f"Метаданные: {docs[0].metadata}")
            logger.info(f"Контент (первые 500 символов): {docs[0].page_content[:500]}...")

    except Exception as e:
        logger.error(f"❌ Ошибка загрузки CSV: {e}")
        import traceback
        logger.error(traceback.format_exc())

    return docs

Writing /content/data_loader.py


In [6]:
%%writefile /content/database.py
import os
import shutil
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from config import logger, DB_PATH, DB_BACKUP_PATH, EMBEDDING_MODEL, HR_COLLECTION_NAME
from data_loader import load_resumes_csv

# Инициализация эмбеддингов
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

def get_db(collection_name):
    """Возвращает подключение к векторной базе"""
    return Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=DB_PATH
    )

def restore_db_from_drive():
    """Восстанавливает базу с Google Диска, если она там есть"""
    if os.path.exists(DB_BACKUP_PATH):
        logger.info("📦 Нашел резервную копию базы на Диске! Восстанавливаю...")
        try:
            shutil.unpack_archive(DB_BACKUP_PATH, DB_PATH, 'zip')
            logger.info("✅ База восстановлена. Индексация не требуется.")
            return True
        except Exception as e:
            logger.error(f"❌ Ошибка восстановления базы: {e}")

    return False

def backup_db_to_drive():
    """Сохраняет базу на Google Диск"""
    try:
        logger.info("💾 Сохраняю базу данных на Google Диск...")
        base_name = DB_BACKUP_PATH.replace(".zip", "")
        shutil.make_archive(base_name, 'zip', DB_PATH)
        logger.info("✅ Бэкап сохранен!")
    except Exception as e:
        logger.error(f"❌ Ошибка сохранения бэкапа: {e}")

def init_db_if_empty():
    """Инициализирует векторную базу данных"""

    # 1. Сначала пробуем восстановить из архива
    if restore_db_from_drive():
        return

    # 2. Если архива нет - создаем с нуля
    db_hr = get_db(HR_COLLECTION_NAME)

    updated = False

    # Проверяем, есть ли уже данные
    try:
        count = db_hr._collection.count()
        logger.info(f"📊 Текущее количество документов в базе: {count}")

        if count == 0:
            logger.info("⚠️ База пуста. Начинаю индексацию резюме...")
            data = load_resumes_csv()

            if data:
                # Добавляем документы пачками по 100
                batch_size = 100
                for i in tqdm(range(0, len(data), batch_size), desc="Индексация резюме"):
                    batch = data[i:i + batch_size]
                    db_hr.add_documents(batch)

                updated = True
                logger.info(f"✅ Индексация завершена. Добавлено {len(data)} документов.")
            else:
                logger.warning("⚠️ Нет данных для индексации.")
        else:
            logger.info("✅ База уже содержит данные. Пропускаю индексацию.")

    except Exception as e:
        logger.error(f"❌ Ошибка при работе с базой: {e}")
        # Если ошибка, пробуем пересоздать
        if os.path.exists(DB_PATH):
            shutil.rmtree(DB_PATH)
        # Инициализируем заново
        db_hr = get_db(HR_COLLECTION_NAME)
        data = load_resumes_csv()
        if data:
            for i in tqdm(range(0, len(data), 100), desc="Индексация резюме"):
                db_hr.add_documents(data[i:i+100])
            updated = True

    # 3. Сохраняем на Диск если обновили
    if updated:
        backup_db_to_drive()

def force_reload_db():
    """Принудительно перезагружает базу данных"""
    logger.info("🔄 Принудительная перезагрузка базы данных...")

    # Удаляем старую базу
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)

    # Удаляем бэкап если есть
    if os.path.exists(DB_BACKUP_PATH):
        os.remove(DB_BACKUP_PATH)

    # Инициализируем заново
    init_db_if_empty()

    # Проверяем результат
    db_hr = get_db(HR_COLLECTION_NAME)
    count = db_hr._collection.count()
    logger.info(f"✅ База перезагружена. Количество документов: {count}")

    return count

Writing /content/database.py


In [7]:
%%writefile /content/chains.py
import asyncio
import re
from langchain_ollama import ChatOllama
from config import OLLAMA_MODEL, logger, HR_COLLECTION_NAME
from database import get_db

# --- НАСТРОЙКИ ---
HR_FETCH_K = 30  # Сколько кандидатов искать изначально
HR_FINAL_K = 6   # Сколько показывать в итоге

# Нейросеть с низкой температурой для стабильности
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.1)

# --- ВЕКТОРНАЯ БАЗА ---
def get_hr_db():
    return get_db(HR_COLLECTION_NAME)

# --- УТИЛИТЫ ДЛЯ ФОРМАТИРОВАНИЯ ---
def extract_field(content, field_name):
    """Извлекает значение поля из текста резюме"""
    pattern = f"{re.escape(field_name)} (.*?)(\\n|$)"
    match = re.search(pattern, content, re.IGNORECASE)
    return match.group(1).strip() if match else "Не указано"

def format_hr_source(doc, index):
    """Форматирует одно резюме для отображения"""
    content = doc.page_content
    metadata = doc.metadata

    # Извлекаем основные поля
    position = extract_field(content, "🎯 ЖЕЛАЕМАЯ ДОЛЖНОСТЬ:")
    if position == "Не указано":
        position = extract_field(content, "📌 ПРОФЕССИЯ:")

    salary = extract_field(content, "💰 ЗАРПЛАТА:")
    experience = extract_field(content, "📅 ОПЫТ РАБОТЫ:")
    skills = extract_field(content, "🛠 НАВЫКИ:")

    # Обрезаем длинные навыки
    if len(skills) > 80:
        skills = skills[:80] + "..."

    # Ссылка
    link_match = re.search(r"🔗 Ссылка: (http\S+)", content)
    link = link_match.group(1) if link_match else ""

    # Формируем блок
    result = f"**{index}. {position}**"

    if experience != "Не указано":
        result += f"\n⏳ {experience}"

    if salary != "Не указано":
        result += f"\n💰 {salary}"

    if skills != "Не указано":
        result += f"\n🛠 {skills}"

    if link:
        result += f"\n🔗 [Резюме]({link})"

    # Добавляем ID для отладки
    resume_id = metadata.get('id', '')
    if resume_id:
        result += f" `[ID: {resume_id[-6:]}]`"

    return result

# --- ПРОЦЕСС ПОИСКА С НЕЙРОСЕТЬЮ ---
async def run_hr_query(criteria):
    """
    Выполняет поиск кандидатов по критериям с использованием нейросети.
    criteria: словарь с полями position, salary, experience
    """

    # Строим текстовый запрос для семантического поиска
    query_parts = []

    if criteria.get('position') and criteria['position'] != "Другое":
        query_parts.append(f"должность {criteria['position']}")

    if criteria.get('salary') and criteria['salary'] != "Не важно":
        query_parts.append(f"зарплата {criteria['salary']}")

    if criteria.get('experience') and criteria['experience'] != "Не важно":
        query_parts.append(f"опыт работы {criteria['experience']}")

    query_text = " ".join(query_parts) if query_parts else "кандидат резюме"

    logger.info(f"🔍 Семантический поиск по запросу: {query_text}")

    # Получаем векторную базу
    db = get_hr_db()

    # Ищем семантически похожие резюме
    try:
        raw_results = db.similarity_search(query_text, k=HR_FETCH_K)
    except Exception as e:
        logger.error(f"Ошибка при поиске в базе: {e}")
        return "❌ Ошибка при поиске в базе данных."

    if not raw_results:
        return "😔 *Не найдено подходящих кандидатов.*\n\nПопробуйте изменить критерии поиска."

    # Фильтруем результаты по критериям (пост-фильтрация)
    filtered_results = []

    for doc in raw_results:
        content = doc.page_content.lower()
        metadata = doc.metadata

        # Проверка по должности
        position_ok = True
        if criteria.get('position') and criteria['position'] != "Другое":
            position = criteria['position'].lower()
            # Ищем в profession и desired_position
            profession = metadata.get('profession', '').lower()
            desired_position = metadata.get('desired_position', '').lower()

            # Более гибкий поиск: ищем частичное совпадение
            if (position not in profession and
                position not in desired_position and
                not any(word in profession for word in position.split()) and
                not any(word in desired_position for word in position.split())):
                position_ok = False

        # Проверка по зарплате
        salary_ok = True
        if criteria.get('salary') and criteria['salary'] != "Не важно":
            salary_filter = criteria['salary']
            salary_value = metadata.get('salary_value', '')
            has_salary = metadata.get('has_salary_value', 'False') == 'True'

            if has_salary and salary_value:
                try:
                    # Пытаемся извлечь число из зарплаты
                    import re
                    numbers = re.findall(r'\d+', salary_value)
                    if numbers:
                        salary_num = float(numbers[0])

                        # Проверяем диапазон
                        if salary_filter == "до 50000 ₽" and salary_num > 50000:
                            salary_ok = False
                        elif salary_filter == "50000-100000 ₽" and not (50000 <= salary_num <= 100000):
                            salary_ok = False
                        elif salary_filter == "100000-150000 ₽" and not (100000 <= salary_num <= 150000):
                            salary_ok = False
                        elif salary_filter == "150000-200000 ₽" and not (150000 <= salary_num <= 200000):
                            salary_ok = False
                        elif salary_filter == "200000-300000 ₽" and not (200000 <= salary_num <= 300000):
                            salary_ok = False
                        elif salary_filter == "выше 300000 ₽" and salary_num <= 300000:
                            salary_ok = False
                except:
                    pass

        # Проверка по опыту
        experience_ok = True
        if criteria.get('experience') and criteria['experience'] != "Не важно":
            exp_filter = criteria['experience'].lower()
            exp_value = metadata.get('work_experience_years', '').lower()
            exp_bucket = metadata.get('experience_bucket', '').lower()

            # Маппинг опыта
            exp_map = {
                "нет опыта": ["0", "нет", "без опыта"],
                "до 1 года": ["0.5", "0.3", "0.8", "меньше года"],
                "1-3 года": ["1", "1.5", "2", "2.5", "3", "1-3"],
                "3-5 лет": ["3", "3.5", "4", "4.5", "5", "3-5"],
                "5-10 лет": ["5", "6", "7", "8", "9", "10", "5-10"],
                "более 10 лет": ["10", "11", "12", "15", "20", "более"]
            }

            search_terms = exp_map.get(exp_filter, [])

            # Проверяем опыт
            found = False
            for term in search_terms:
                if term in exp_value or term in exp_bucket:
                    found = True
                    break

            if not found:
                experience_ok = False

        # Если все критерии проходят, добавляем в результаты
        if position_ok and salary_ok and experience_ok:
            filtered_results.append(doc)

    # Берем топ результатов
    final_results = filtered_results[:HR_FINAL_K]

    if not final_results:
        return "😔 *После фильтрации по вашим критериям не осталось кандидатов.*\n\nПопробуйте выбрать 'Не важно' для некоторых параметров."

    # Формируем контекст для нейросети
    context = ""
    for i, doc in enumerate(final_results):
        context += f"=== КАНДИДАТ {i+1} ===\n{doc.page_content}\n\n"

    # Промпт для нейросети (аналогично исходной работе)
    prompt = f"""
    [SYSTEM: ВЫ СТАРШИЙ HR-РЕКРУТЕР. ОТВЕЧАЙТЕ ТОЛЬКО НА РУССКОМ ЯЗЫКЕ.]

    КРИТЕРИИ ПОИСКА ПОЛЬЗОВАТЕЛЯ:
    - Должность: {criteria.get('position', 'Не указано')}
    - Зарплата: {criteria.get('salary', 'Не указано')}
    - Опыт: {criteria.get('experience', 'Не указано')}

    ТОП КАНДИДАТОВ:
    {context}

    ЗАДАЧА:
    1. Проанализируйте критерии пользователя.
    2. Выберите ОДНОГО лучшего кандидата, который лучше всего соответствует требованиям.
       - Приоритет: явное соответствие зарплате и опыту.
       - Учитывайте навыки и опыт.
    3. Напишите короткую рекомендацию с обоснованием.

    ФОРМАТ ВЫВОДА:
    "✅ **Рекомендую кандидата №[N]** ([Краткая должность])
    📊 **Обоснование:** [Объяснение на русском языке]"
    """

    try:
        # Запрашиваем ответ у нейросети
        response = await llm.ainvoke(prompt)
        ai_response = response.content

        # Форматируем список кандидатов
        candidates_list = "\n".join([format_hr_source(doc, i+1) for i, doc in enumerate(final_results)])

        # Формируем итоговый ответ
        result = f"{ai_response}\n\n___\n🔍 *НАЙДЕННЫЕ КАНДИДАТЫ ({len(final_results)}):*\n{candidates_list}"

        return result

    except Exception as e:
        logger.error(f"Ошибка нейросети: {e}")
        # Если нейросеть не сработала, показываем просто список
        candidates_list = "\n".join([format_hr_source(doc, i+1) for i, doc in enumerate(final_results)])
        return f"✅ *Найдено {len(final_results)} кандидатов:*\n\n{candidates_list}"

Writing /content/chains.py


In [8]:
%%writefile /content/main.py
import nest_asyncio
nest_asyncio.apply()

import logging
import asyncio
import os
import random
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from config import TELEGRAM_TOKEN, SAMPLE_HR_QUERIES
from chains import run_hr_query
from database import init_db_if_empty, force_reload_db
import keyboards as kb

# Настройка логов
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger("HR_Researcher_Bot")

# Состояния диалога
class ConversationState:
    START = 0
    ASK_POSITION = 1
    ASK_SALARY = 2
    ASK_EXPERIENCE = 3
    SEARCHING = 4

# Глобальная переменная для хранения пользовательских данных
user_sessions = {}

# --- ОБРАБОТЧИКИ КОМАНД ---
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработчик команды /start"""
    user_id = update.effective_user.id
    user_sessions[user_id] = {
        'state': ConversationState.START,
        'criteria': {},
        'step': 0
    }

    welcome_text = """
👋 *Добро пожаловать в AI HR Researcher!*

Я помогу вам найти идеальных кандидатов из базы резюме.

📊 *База данных:* 1000+ резюме
🧠 *Использую нейросеть* для анализа кандидатов
🔍 *Процесс поиска:* 3 вопроса
⏱ *Время поиска:* 10-20 секунд

👇 *Выберите действие:*
    """

    # ИСПРАВЛЕНИЕ: используем effective_message вместо message
    if update.effective_message:
        await update.effective_message.reply_text(
            welcome_text,
            reply_markup=kb.get_main_keyboard(),
            parse_mode="Markdown"
        )

async def help_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработчик команды /help"""
    help_text = """
ℹ️ *Справка по использованию бота*

*Основные функции:*
• *Найти кандидата* — запустить пошаговый поиск (3 вопроса)
• *Обновить базу* — перезагрузить векторную базу данных
• *Случайный запрос* — пример быстрого поиска

*Процесс поиска (3 вопроса):*
1️⃣ Выберите должность (Backend, Frontend, DevOps разработчик или IT project manager)
2️⃣ Укажите зарплатный диапазон
3️⃣ Выберите опыт работы

*Как работает поиск:*
1. Я ищу семантически похожие резюме
2. Фильтрую по вашим критериям
3. Нейросеть анализирует лучших кандидатов
4. Вы получаете рекомендацию с обоснованием

*Управление поиском:*
• *Отменить поиск* — вернуться в главное меню
• */start* — перезапустить бота
• */help* — показать эту справку

*Советы:*
• Используйте кнопки для выбора ответов
• Для любого вопроса можно выбрать "Не важно"
• Нейросеть учитывает не только точные совпадения, но и похожие навыки
    """

    if update.effective_message:
        await update.effective_message.reply_text(
            help_text,
            parse_mode="Markdown",
            reply_markup=kb.get_main_keyboard()
        )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработчик всех текстовых сообщений"""
    if not update.effective_message:
        return

    user_id = update.effective_user.id
    text = update.effective_message.text

    # Инициализация сессии для нового пользователя
    if user_id not in user_sessions:
        user_sessions[user_id] = {
            'state': ConversationState.START,
            'criteria': {},
            'step': 0
        }

    session = user_sessions[user_id]

    # --- ОБРАБОТКА ГЛАВНОГО МЕНЮ ---
    if text == "👨‍💻 Найти кандидата":
        session['state'] = ConversationState.ASK_POSITION
        session['criteria'] = {}
        session['step'] = 1

        await update.effective_message.reply_text(
            "🎯 *Шаг 1/3: Выбор должности*\n\n"
            "Какую должность вы ищете?\n"
            "Выберите из списка или напишите свой вариант:",
            reply_markup=kb.get_position_keyboard(),
            parse_mode="Markdown"
        )
        return

    elif text == "🔄 Обновить базу":
        msg = await update.effective_message.reply_text("🔄 Перезагружаю векторную базу данных...")
        try:
            count = force_reload_db()
            await msg.edit_text(f"✅ Векторная база успешно обновлена!\n📊 Проиндексировано резюме: {count}")
        except Exception as e:
            await msg.edit_text(f"❌ Ошибка при обновлении базы:\n{str(e)}")
        return

    elif text == "🎲 Случайный запрос":
        random_position = random.choice(SAMPLE_HR_QUERIES)
        session['criteria'] = {
            'position': random_position,
            'salary': "Не важно",
            'experience': "Не важно"
        }

        await update.effective_message.reply_text(
            f"🎲 *Случайный запрос:* {random_position}\n\n"
            f"🔍 Ищу кандидатов...",
            parse_mode="Markdown"
        )

        await process_search(update, session['criteria'])
        session['state'] = ConversationState.START
        return

    elif text == "❌ Отменить поиск":
        session['state'] = ConversationState.START
        session['criteria'] = {}

        await update.effective_message.reply_text(
            "❌ Поиск отменен.\nВы вернулись в главное меню.",
            reply_markup=kb.get_main_keyboard()
        )
        return

    # --- ОБРАБОТКА ДИАЛОГА ПОИСКА ---

    # Шаг 1: Должность
    if session['state'] == ConversationState.ASK_POSITION:
        session['criteria']['position'] = text
        session['state'] = ConversationState.ASK_SALARY
        session['step'] = 2

        await update.effective_message.reply_text(
            f"✅ *Должность сохранена:* {text}\n\n"
            "💰 *Шаг 2/3: Зарплатный диапазон*\n\n"
            "Какой диапазон зарплаты интересует?\n"
            "Если зарплата не важна, выберите 'Не важно':",
            reply_markup=kb.get_salary_keyboard(),
            parse_mode="Markdown"
        )

    # Шаг 2: Зарплата
    elif session['state'] == ConversationState.ASK_SALARY:
        session['criteria']['salary'] = text
        session['state'] = ConversationState.ASK_EXPERIENCE
        session['step'] = 3

        await update.effective_message.reply_text(
            f"✅ *Зарплата сохранена:* {text}\n\n"
            "📅 *Шаг 3/3: Опыт работы*\n\n"
            "Какой опыт работы требуется?\n"
            "Если опыт не важен, выберите 'Не важно':",
            reply_markup=kb.get_experience_keyboard(),
            parse_mode="Markdown"
        )

    # Шаг 3: Опыт
    elif session['state'] == ConversationState.ASK_EXPERIENCE:
        session['criteria']['experience'] = text
        session['state'] = ConversationState.SEARCHING

        # Формируем сводку
        criteria = session['criteria']
        summary = "📋 *Сводка ваших критериев:*\n\n"

        steps = [
            ("Должность", criteria.get('position', 'Не указано')),
            ("Зарплата", criteria.get('salary', 'Не указано')),
            ("Опыт", criteria.get('experience', 'Не указано'))
        ]

        for name, value in steps:
            summary += f"• *{name}:* {value}\n"

        summary += "\n🔍 *Запускаю нейросеть для поиска...*"

        await update.effective_message.reply_text(
            summary,
            parse_mode="Markdown"
        )

        # Запускаем поиск с нейросетью
        await process_search(update, criteria)

        # Сбрасываем сессию
        session['state'] = ConversationState.START
        session['criteria'] = {}

    # Обработка команды помощи в любом состоянии
    elif text == "ℹ️ Помощь":
        await help_cmd(update, context)

async def process_search(update: Update, criteria):
    """Выполняет поиск с нейросетью и показывает результаты"""
    if not update.effective_message:
        return

    # Показываем статус "печатает"
    await update.effective_chat.send_action("typing")

    # Сообщение о начале поиска
    search_msg = await update.effective_message.reply_text(
        "🔍 *Ищу кандидатов в базе...*\n"
        "_Нейросеть анализирует резюме (это может занять 10-20 секунд)_",
        parse_mode="Markdown"
    )

    try:
        # Выполняем поиск с нейросетью
        result = await run_hr_query(criteria)

        # Отправляем результаты
        await search_msg.edit_text(
            result,
            parse_mode="Markdown",
            disable_web_page_preview=True
        )

        # Предлагаем новый поиск
        await update.effective_message.reply_text(
            "🔄 *Хотите сделать новый поиск?*\n"
            "Используйте кнопку '👨‍💻 Найти кандидата'",
            reply_markup=kb.get_main_keyboard(),
            parse_mode="Markdown"
        )

    except Exception as e:
        logger.error(f"Ошибка поиска: {e}")
        await search_msg.edit_text(
            f"❌ *Произошла ошибка при поиске:*\n"
            f"`{str(e)[:200]}`\n\n"
            f"Попробуйте обновить базу данных или изменить критерии.",
            parse_mode="Markdown"
        )

# --- ЗАПУСК БОТА ---
def run_bot():
    """Функция для запуска бота"""
    print("=" * 50)
    print("🤖 ЗАПУСК AI HR RESEARCHER BOT")
    print("=" * 50)

    # 1. Инициализация базы данных
    print("📂 Инициализация векторной базы данных...")
    try:
        init_db_if_empty()
        print("✅ Векторная база готова")
    except Exception as e:
        print(f"❌ Ошибка инициализации базы: {e}")

    # 2. Проверка токена
    if TELEGRAM_TOKEN == "ВАШ_ТОКЕН_БОТА_ЗДЕСЬ":
        print("❌ ОШИБКА: Токен бота не установлен!")
        print("ℹ️ Замените токен в файле config.py")
        return

    print(f"🔑 Токен бота: {TELEGRAM_TOKEN[:10]}...")

    # 3. Создание приложения
    print("🚀 Создаю приложение бота...")
    app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()

    # 4. Добавление обработчиков
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_cmd))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    # 5. Запуск бота
    print("🟢 Запускаю бота...")
    print("📱 Перейдите в Telegram и начните диалог с ботом")
    print("⏸ Для остановки нажмите Ctrl+C")
    print("=" * 50)

    # Запускаем бота
    app.run_polling(allowed_updates=Update.ALL_TYPES)

if __name__ == "__main__":
    run_bot()

Writing /content/main.py


In [9]:
import sys
import os

# Добавляем текущую директорию в путь Python
sys.path.insert(0, '/content')

print("🔍 Проверяю конфигурацию...")

# Проверяем наличие файлов
required_files = ['config.py', 'keyboards.py', 'normalizer.py', 'data_loader.py',
                  'database.py', 'chains.py', 'main.py']
for file in required_files:
    if os.path.exists(f'/content/{file}'):
        print(f"✅ {file} найден")
    else:
        print(f"❌ {file} не найден")

# Проверяем токен
try:
    from config import TELEGRAM_TOKEN, RESUME_FILE

    print(f"\n🔑 Токен бота: {TELEGRAM_TOKEN[:10]}...")

    # Проверяем наличие файла с резюме
    print(f"\n📁 Проверяю файл с резюме: {RESUME_FILE}")

    if os.path.exists(RESUME_FILE):
        print("✅ Файл с резюме найден")

        # Показываем информацию о файле
        import pandas as pd
        try:
            print("📖 Читаю файл...")
            df = pd.read_csv(RESUME_FILE, encoding='utf-8')
            print(f"📊 Записей в файле: {len(df)}")
            print(f"📋 Колонки: {list(df.columns)}")

            # Показываем первые 2 записи
            print("\n📝 Первые 2 записи:")
            for i in range(min(2, len(df))):
                print(f"\n--- Запись {i+1} ---")
                print(f"profession: {df.iloc[i].get('profession', 'Нет данных')}")
                print(f"desired_position: {df.iloc[i].get('desired_position', 'Нет данных')}")
                print(f"work_experience_years: {df.iloc[i].get('work_experience_years', 'Нет данных')}")
                print(f"salary_value: {df.iloc[i].get('salary_value', 'Нет данных')}")

        except UnicodeDecodeError:
            print("🔄 Пробую кодировку cp1251...")
            df = pd.read_csv(RESUME_FILE, encoding='cp1251')
            print(f"✅ Файл прочитан с кодировкой cp1251")
            print(f"📊 Записей: {len(df)}")
            print(f"📋 Колонки: {list(df.columns)}")
        except Exception as e:
            print(f"❌ Не удалось прочитать файл: {e}")
    else:
        print(f"❌ Файл с резюме не найден по пути: {RESUME_FILE}")
        print("\n🔍 Ищу файл в других местах...")

        # Ищем файл в других возможных местах
        search_paths = [
            "/content/drive/MyDrive/researcher/researcher_1000samples.csv",
            "/content/drive/researcher/researcher_1000samples.csv",
            "/content/drive/MyDrive/researcher_1000samples.csv",
        ]

        found = False
        for path in search_paths:
            if os.path.exists(path):
                print(f"✅ Файл найден по пути: {path}")
                print(f"📝 Измените путь в config.py на: RESUME_FILE = \"{path}\"")
                found = True
                break

        if not found:
            print("❌ Файл не найден ни в одном из возможных мест")
            print("📂 Проверьте наличие файла в Google Drive")

except ImportError as e:
    print(f"❌ Ошибка импорта: {e}")
    print("ℹ️ Перезапустите все ячейки с начала")

🔍 Проверяю конфигурацию...
✅ config.py найден
✅ keyboards.py найден
✅ normalizer.py найден
✅ data_loader.py найден
✅ database.py найден
✅ chains.py найден
✅ main.py найден

🔑 Токен бота: 8379355384...

📁 Проверяю файл с резюме: /content/drive/MyDrive/researcher/researcher_1000samples.csv
✅ Файл с резюме найден
📖 Читаю файл...
📊 Записей в файле: 970
📋 Колонки: ['profession', 'desired_position', 'work_experience_years', 'has_work_experience_years', 'experience_bucket', 'salary_value', 'has_salary_value', 'is_male', 'age', 'location', 'education_level', 'education_facility', 'languages', 'english_level', 'skills', 'self_description', 'has_emoji', 'resume_id', 'url', 'text_raw_enriched', 'text_lemmatized_enriched', 'desc_verb_count', 'desc_noun_count']

📝 Первые 2 записи:

--- Запись 1 ---
profession: frontend
desired_position: frontend-developer
work_experience_years: 1.5
salary_value: 70000.0

--- Запись 2 ---
profession: it project manager
desired_position: менеджер it-проектов
work_exp

In [10]:
import sys
sys.path.insert(0, '/content')

print("🚀 Запускаю AI HR Researcher Bot...")
print("=" * 50)

try:
    # Импортируем функцию run_bot из main
    from main import run_bot
    run_bot()

except KeyboardInterrupt:
    print("\n✅ Бот остановлен пользователем")

except Exception as e:
    print(f"\n❌ Ошибка при запуске: {e}")
    import traceback
    print(traceback.format_exc())
    print("\n🔧 Возможные решения:")
    print("1. Проверьте токен бота в config.py")
    print("2. Убедитесь, что файл с резюме существует")
    print("3. Проверьте путь в config.py (RESUME_FILE)")
    print("4. Перезапустите runtime: Runtime → Restart runtime")
    print("5. Проверьте, что Ollama запущен")

🚀 Запускаю AI HR Researcher Bot...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🤖 ЗАПУСК AI HR RESEARCHER BOT
📂 Инициализация векторной базы данных...


Индексация резюме: 100%|██████████| 10/10 [02:32<00:00, 15.26s/it]


✅ Векторная база готова
🔑 Токен бота: 8379355384...
🚀 Создаю приложение бота...
🟢 Запускаю бота...
📱 Перейдите в Telegram и начните диалог с ботом
⏸ Для остановки нажмите Ctrl+C


ERROR:HR_Researcher_Bot:Ошибка нейросети: Server disconnected without sending a response.



❌ Ошибка при запуске: Cannot close a running event loop
Traceback (most recent call last):
  File "/tmp/ipython-input-3641807052.py", line 10, in <cell line: 0>
    run_bot()
  File "/content/main.py", line 324, in run_bot
    app.run_polling(allowed_updates=Update.ALL_TYPES)
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py", line 839, in run_polling
    return self.__run(
           ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py", line 1084, in __run
    loop.close()
  File "/usr/lib/python3.12/asyncio/unix_events.py", line 68, in close
    super().close()
  File "/usr/lib/python3.12/asyncio/selector_events.py", line 101, in close
    raise RuntimeError("Cannot close a running event loop")
RuntimeError: Cannot close a running event loop


🔧 Возможные решения:
1. Проверьте токен бота в config.py
2. Убедитесь, что файл с резюме существует
3. Проверьте путь в config.py (RESUME_FILE)
4. Перезапустите runtime: Runtime →